# Navigating Without GPS: Magnetic Anomaly Navigation (MagNav) in Python

This notebook follows the U.S. Air Force / MIT **MagNav** tutorial workflow
(AFIT 2023 PLANS MagNav Tutorial + the open-source MagNav.jl package) and rebuilds it in Python.

**Part 1:** Use real Air Force flight data to clean up the aircraft's own magnetic noise
with the **Tolles-Lawson** model.
**Part 2:** Use a real magnetic anomaly map to navigate with GPS switched off, and compare
that against an inertial navigation system (INS) drifting by itself.

Data: MagNav.jl test data (public domain, U.S. Department of the Air Force).

In [ ]:
# Setup: download the data (runs in Google Colab, no installs needed)
import os, urllib.request
import numpy as np
import scipy.io as sio
from scipy.signal import butter, filtfilt
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/MIT-AI-Accelerator/MagNav.jl/master/test/test_data/"
FILES = ["test_data_TL.mat", "test_data_params.mat", "test_data_map.mat"]
DATA_DIR = os.environ.get("MAGNAV_DATA", "data")
os.makedirs(DATA_DIR, exist_ok=True)
for f in FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE + f, path)
        print("downloaded", f)

def load(name, key):
    return sio.loadmat(os.path.join(DATA_DIR, name), squeeze_me=True, struct_as_record=False)[key]

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3})
FIG_DIR = os.environ.get("MAGNAV_FIGS", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

## Part 1: Cleaning the plane's magnetic noise (Tolles-Lawson)

A magnetometer on a plane measures **Earth's field + the plane's own field**. The plane's field
comes from three things (Tolles & Lawson, 1950):

1. **Permanent**: magnetized steel parts (3 terms)
2. **Induced**: metal that becomes magnetic inside Earth's field (6 terms)
3. **Eddy currents**: currents in the metal skin when the plane turns (9 terms)

That gives **18 terms**. Each term is something we can compute from a *vector* magnetometer
(the fluxgate, which measures Bx, By, Bz). We then find 18 coefficients **c** so that

    plane_noise ≈ A · c

using least squares on a calibration flight (the plane flies a box and does pitch, roll and yaw moves).

In [ ]:
# Load real calibration-flight data
T = load("test_data_TL.mat", "TL_data")
P = load("test_data_params.mat", "params").TL
fs = float(T.fs)
Bx, By, Bz = T.flux_a_x, T.flux_a_y, T.flux_a_z   # fluxgate (vector) magnetometer [nT]
mag_uc = T.mag_1_uc                               # scalar magnetometer, uncompensated [nT]
t = np.arange(len(mag_uc)) / fs
print(f"{len(mag_uc)} samples at {fs:.0f} Hz = {t[-1]:.0f} seconds of flight")

In [ ]:
# Build the 18-column Tolles-Lawson A matrix
def fdm(x):
    """Central finite difference (rate of change per sample)."""
    d = np.empty_like(x)
    d[0], d[-1] = x[1] - x[0], x[-1] - x[-2]
    d[1:-1] = (x[2:] - x[:-2]) / 2
    return d

def tolles_lawson_A(Bx, By, Bz, Bt_scale=50000):
    Bt = np.sqrt(Bx**2 + By**2 + Bz**2)
    cx, cy, cz = Bx / Bt, By / Bt, Bz / Bt            # direction cosines
    B, Bdot, c = [Bx, By, Bz], [fdm(Bx), fdm(By), fdm(Bz)], [cx, cy, cz]
    permanent = [cx, cy, cz]
    induced = [cx*Bx, cx*By, cx*Bz, cy*By, cy*Bz, cz*Bz]
    eddy = [c[i] * Bdot[j] for i in range(3) for j in range(3)]
    cols = permanent + [v / Bt_scale for v in induced] + [v / Bt_scale for v in eddy]
    return np.column_stack(cols)

A = tolles_lawson_A(Bx, By, Bz, P.Bt_scale)
print("A matrix shape:", A.shape)
print("Matches MagNav.jl's A matrix:", np.allclose(A, T.A_a))

In [ ]:
# Fit the coefficients (bandpass filter, then ridge regression)
# The bandpass (0.1 to 0.9 Hz) keeps the fast wiggles caused by the plane's movement and
# removes the slow changes that come from the ground's real magnetic anomalies.
b, a = butter(4, [P.pass1, P.pass2], btype="band", fs=fs)
trim = int(P.trim)
bp = lambda x: filtfilt(b, a, x)
A_f = np.column_stack([bp(A[:, i]) if A[:, i].std() > 1e-12 else A[:, i] for i in range(A.shape[1])])[trim:-trim]
y_f = bp(mag_uc)[trim:-trim]
lam = getattr(P, "lambda")
coef = np.linalg.solve(A_f.T @ A_f + lam * np.eye(18), A_f.T @ y_f)

plane_field = A @ coef
mag_c = mag_uc - plane_field
print("Difference vs MagNav.jl's compensated signal (std, nT):", round(np.std(mag_c - T.mag_1_c), 3))

In [ ]:
# How much noise did we remove?
hp = lambda x: filtfilt(b, a, x)  # look only at the manoeuvre band
before, after = np.std(hp(mag_uc)[trim:-trim]), np.std(hp(mag_c)[trim:-trim])
print(f"Plane's own field removed: {np.std(plane_field):.2f} nT std (offset {plane_field.mean():.1f} nT)")
print(f"Manoeuvre noise before: {before:.2f} nT  after: {after:.2f} nT  ->  {before/after:.1f}x cleaner")

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(t, mag_uc - mag_uc.mean(), label="raw (uncompensated)", color="#c0392b", lw=1)
ax[0].plot(t, mag_c - mag_c.mean(), label="after Tolles-Lawson", color="#2471a3", lw=1)
ax[0].set_ylabel("field − mean [nT]"); ax[0].legend(); ax[0].set_title("Real Air Force calibration flight")
ax[1].plot(t, hp(mag_uc), color="#c0392b", lw=1, label="raw: manoeuvre band")
ax[1].plot(t, hp(mag_c), color="#2471a3", lw=1, label="compensated: manoeuvre band")
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("nT"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/1_tolles_lawson.png"); plt.show()

In [ ]:
# Which of the 18 terms matter most?
names = ["perm x", "perm y", "perm z", "ind xx", "ind xy", "ind xz", "ind yy", "ind yz", "ind zz"] + \
        [f"eddy {i}{j}" for i in "xyz" for j in "xyz"]
contrib = np.std(A * coef, axis=0)
plt.figure(figsize=(9, 3.5))
plt.bar(names, contrib, color=["#7d3c98"]*3 + ["#17a589"]*6 + ["#d68910"]*9)
plt.xticks(rotation=60); plt.ylabel("contribution [nT]"); plt.title("How much each Tolles-Lawson term contributes")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/2_terms.png"); plt.show()

## Part 2: Navigating with GPS turned off

Earth's crust has a magnetic "fingerprint" (anomalies from rocks underground). If we have a
**map** of it, we can compare what the magnetometer reads against the map and work out where we are.

Setup:
- **Map:** real magnetic anomaly map (100 × 100 grid, ~87 km × 111 km, at 305 m altitude) from MagNav.jl.
- **Truth:** a simulated 25-minute flight across the map.
- **INS:** a backup navigation system that drifts slowly (small sensor errors add up).
- **Magnetometer:** reads the map value at the true position plus noise. The noise level is the
  leftover noise from Part 1, so the two halves are linked.
- **Particle filter:** thousands of "guesses" about how far the INS has drifted. Guesses whose
  predicted map reading matches the real reading survive; bad ones die off.

In [ ]:
# Load the magnetic anomaly map
M = load("test_data_map.mat", "map_data")
grid = np.asarray(M.map, float)
ny, nx = grid.shape
xs = np.arange(nx) * float(M.dx)   # east [m]
ys = np.arange(ny) * float(M.dy)   # north [m]
mag_map = RegularGridInterpolator((ys, xs), grid, bounds_error=False, fill_value=np.nan)
print(f"Map: {xs[-1]/1000:.0f} km × {ys[-1]/1000:.0f} km, anomalies from {grid.min():.0f} to {grid.max():.0f} nT")

In [ ]:
# Simulate the true flight path and a drifting INS
rng = np.random.default_rng(7)
dt, duration, speed = 1.0, 1500, 60.0            # 1 Hz, 25 min, 60 m/s (~215 km/h, like a Cessna)
n = int(duration / dt)
heading = np.deg2rad(40) + np.cumsum(np.where((np.arange(n) // 300) % 2 == 0, 0.0015, -0.0015))
true = np.zeros((n, 2)); true[0] = [12_000, 12_000]
vel = speed * np.column_stack([np.sin(heading), np.cos(heading)])   # (east, north)
true[1:] = true[0] + np.cumsum(vel[:-1] * dt, axis=0)

# INS error model: a small velocity bias plus random-walk velocity error
# (tune INS_DRIFT to compare cheaper vs better INS units)
INS_DRIFT = 1.0
v_err = INS_DRIFT * (np.array([0.35, -0.25]) + np.cumsum(rng.normal(0, 0.02, (n, 2)), axis=0))
ins = true + np.cumsum(v_err * dt, axis=0)

# Magnetometer measurement = map at the true position + leftover noise from Part 1 + map error
MEAS_NOISE = float(after)     # nT, from Tolles-Lawson residual
MAP_NOISE = 3.0               # nT, maps are never perfect
meas = mag_map(true[:, ::-1]) + rng.normal(0, np.hypot(MEAS_NOISE, MAP_NOISE), n)

In [ ]:
# Particle filter: estimate the INS drift using map matching
def particle_filter(ins, meas, N=10000, init_std=150.0, q=15.0, sigma=None, seed=1):
    r = np.random.default_rng(seed)
    sigma = sigma or np.hypot(MEAS_NOISE, MAP_NOISE) * 2   # a bit wider than the true noise, so the filter is not overconfident
    err = r.normal(0, init_std, (N, 2))         # each particle = a guess of INS drift (east, north)
    w = np.full(N, 1 / N)
    est = np.zeros_like(ins)
    for k in range(len(ins)):
        err += r.normal(0, q, err.shape)                  # spread the guesses a little each second (q metres)
        pos = ins[k] - err                               # where each guess says we really are
        pred = mag_map(pos[:, ::-1])                     # what the map says we should read there
        like = np.exp(-0.5 * ((meas[k] - pred) / sigma) ** 2)
        like[np.isnan(like)] = 0
        w = w * like + 1e-300; w /= w.sum()
        est[k] = ins[k] - w @ err
        if 1 / np.sum(w**2) < N / 2:                     # resample when only a few guesses matter
            idx = r.choice(N, N, p=w); err = err[idx]; w = np.full(N, 1 / N)
    return est

magnav = particle_filter(ins, meas)
err_ins = np.linalg.norm(ins - true, axis=1)
err_mag = np.linalg.norm(magnav - true, axis=1)
dist_km = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(true, axis=0), axis=1))]) / 1000
print(f"After {dist_km[-1]:.0f} km without GPS:")
print(f"  INS alone error: {err_ins[-1]:.0f} m")
print(f"  MagNav error:    {err_mag[-1]:.0f} m   (average over last 5 min: {err_mag[-300:].mean():.0f} m)")
print(f"  -> {err_ins[-1]/err_mag[-1]:.0f}x better")

In [ ]:
# Plot the map and the three paths
fig, ax = plt.subplots(figsize=(7.5, 8))
im = ax.imshow(grid, origin="lower", extent=[0, xs[-1]/1000, 0, ys[-1]/1000], cmap="RdBu_r",
               vmin=-np.percentile(np.abs(grid), 98), vmax=np.percentile(np.abs(grid), 98))
plt.colorbar(im, ax=ax, shrink=0.7, label="magnetic anomaly [nT]")
ax.plot(*true.T / 1000, "k-", lw=2.5, label="true path")
ax.plot(*ins.T / 1000, "--", color="#e67e22", lw=2, label="INS only (drifting)")
ax.plot(*magnav.T / 1000, "-", color="#27ae60", lw=1.8, label="MagNav estimate")
ax.plot(*true[0] / 1000, "k^", ms=10, label="GPS lost here")
ax.set_xlabel("east [km]"); ax.set_ylabel("north [km]"); ax.legend(loc="upper left", fontsize=8)
ax.set_title("Flying on a magnetic map with no GPS"); ax.grid(False)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/3_map_paths.png"); plt.show()

plt.figure(figsize=(9, 3.8))
plt.plot(dist_km, err_ins, color="#e67e22", lw=2, label="INS only")
plt.plot(dist_km, err_mag, color="#27ae60", lw=2, label="MagNav (map matching)")
plt.xlabel("distance flown without GPS [km]"); plt.ylabel("position error [m]")
plt.title("Position error grows for INS, stays bounded for MagNav"); plt.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/4_error.png"); plt.show()

In [ ]:
# Experiment: how much does map quality matter?
# Real maps are blurry and have errors. Q-CTRL's paper lists map quality as a main limitation.
results = {}
for map_err in [1, 3, 10, 25]:
    m = mag_map(true[:, ::-1]) + rng.normal(0, np.hypot(MEAS_NOISE, map_err), n)
    e = np.linalg.norm(particle_filter(ins, m, sigma=2 * np.hypot(MEAS_NOISE, map_err)) - true, axis=1)
    results[map_err] = e
    print(f"map error {map_err:3d} nT -> mean position error {e.mean():5.0f} m, final {e[-1]:5.0f} m")
plt.figure(figsize=(9, 3.8))
for (map_err, e), col in zip(results.items(), ["#1e8449", "#27ae60", "#b7950b", "#8e44ad"]):
    plt.plot(dist_km, e, lw=2, color=col, label=f"map error {map_err} nT")
plt.plot(dist_km, err_ins, color="#e67e22", lw=1.5, ls="--", label="INS only")
plt.xlabel("distance flown without GPS [km]"); plt.ylabel("position error [m]"); plt.legend(fontsize=8)
plt.title("Better maps = better navigation"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/5_map_quality.png"); plt.show()